# UE Data Mining - Projet

**Health**

In [ ]:
# TODO
!git clone https://github.com/Sheik-info/20262027-UE-DataMining-Project.git
%cd 20262027-UE-DataMining-Project

In [ ]:
!git pull



1. Chargement et première exploration du jeu de données

Nous commençons par charger le fichier cardiac_recordings.csv, situé dans le dossier data/health/ du projet.

L’objectif de cette première étape est de comprendre la structure générale du jeu de données avant toute opération de nettoyage :

    le nombre d’observations ;
    le nombre de variables ;
    le contenu des colonnes ;
    les types de données.


In [ ]:
import pandas as pd

df = pd.read_csv("data/health/cardiac_recordings.csv")



Nous vérifions ensuite les dimensions du jeu de données afin de connaître :

    le nombre de lignes ;
    le nombre de colonnes.


In [ ]:
df.shape

Le jeu de données contient 44 observations et 19 variables.

**Aperçu des premières observations**

Nous affichons les premières lignes du jeu de données afin d’obtenir un premier aperçu de son contenu et de comprendre ce que représente une observation.

In [ ]:
df.head()

In [ ]:
df.head()

In [ ]:
%cd /content/20262027-UE-DataMining-Project

In [ ]:
import os

os.listdir("data/health")

In [ ]:
import pandas as pd
import os

for fichier in os.listdir("data/health"):
    df_temp = pd.read_csv("data/health/" + fichier)

    print("\n==============================")
    print(fichier)
    print("Dimensions :", df_temp.shape)
    print("Colonnes :", list(df_temp.columns))

## Nettoyage — `patient_administration.csv`

Ce fichier contient les données administratives des patients (identité, démographie, statut d'enregistrement). C'est la table de référence du jeu de données : tous les autres fichiers (`clinical_encounters`, `clinical_observations`, etc.) y font référence via `patient_number`.

**Problèmes attendus, à vérifier systématiquement :**
1. Doublons de lignes (redondance d'information)
2. Incohérences de format dans les variables catégorielles (casse, orthographe)
3. Plausibilité des valeurs numériques/dates (âges, cohérence chronologique)


In [ ]:
# Chargement du fichier patient_administration depuis le repo cloné
import pandas as pd
pd.set_option('display.max_columns', None)

pa = pd.read_csv('data/health/patient_administration.csv')
print(f"Dimensions initiales : {pa.shape}")
pa.head()

### 1. Détection et suppression des doublons

On vérifie s'il existe des lignes strictement identiques (même patient, mêmes informations répétées), ce qui correspond à une redondance d'information pure (cf. cours 1 — "Gérer les redondances d'informations").

In [ ]:
# Identification des lignes dupliquées (toutes colonnes identiques)
doublons = pa[pa.duplicated(keep=False)]
print(f"Nombre de lignes impliquées dans un doublon : {len(doublons)}")
doublons

In [ ]:
# Suppression du doublon détecté, réinitialisation de l'index
pa = pa.drop_duplicates().reset_index(drop=True)
print(f"Dimensions après suppression des doublons : {pa.shape}")

### 2. Normalisation des variables catégorielles

On inspecte les valeurs uniques des colonnes catégorielles afin de repérer des variantes d'écriture d'une même valeur (ex. différences de casse), qui empêchent un regroupement correct des catégories lors de l'analyse (cf. cours 1 — "Format inconsistant").

In [ ]:
# Inspection des valeurs uniques, avant correction
for col in ['registered_sex', 'record_status', 'home_region']:
    print(f"{col} -> {pa[col].unique()}")

Deux colonnes présentent des incohérences de casse :
- `record_status` : `'active'` et `'ACTIVE'` désignent la même valeur
- `home_region` : `'Valenne Republic'` et `'vALENNE rEPUBLIC'` désignent la même valeur

On normalise ces deux colonnes (minuscules pour `record_status`, casse "titre" pour `home_region`), puis on revérifie que chaque valeur n'apparaît plus que sous une seule forme.

In [ ]:
# Normalisation de la casse sur les deux colonnes concernées
pa['record_status'] = pa['record_status'].str.lower()
pa['home_region'] = pa['home_region'].str.title()

# Vérification post-correction
print(f"record_status -> {pa['record_status'].unique()}")
print(f"home_region -> {pa['home_region'].unique()}")

### 3. Vérification de la plausibilité de l'âge

On calcule l'âge de chaque patient à partir de `date_of_birth`, puis on examine la distribution (moyenne, écart-type, min, max) afin de détecter d'éventuelles valeurs aberrantes (âge négatif, ou supérieur à l'espérance de vie maximale plausible).

In [ ]:
# Conversion de la date de naissance et calcul de l'âge
pa['date_of_birth'] = pd.to_datetime(pa['date_of_birth'])
today = pd.Timestamp.now()
pa['age'] = (today - pa['date_of_birth']).dt.days // 365

print(pa['age'].describe())

Les âges s'étalent entre 23 et 91 ans, avec une moyenne de ~56 ans : aucune valeur n'est aberrante. Aucune correction nécessaire sur cette colonne.

In [ ]:
# Diagnostic préalable : on vérifie que toutes les dates suivent le même format avant conversion
mask_anomalie = ~pa['registration_date'].str.match(r'^\d{4}-\d{2}-\d{2}$')
print(f"Nombre de valeurs avec un format non-standard : {mask_anomalie.sum()}")
pa.loc[mask_anomalie, 'registration_date']

Deux valeurs ne respectent pas le format `YYYY-MM-DD` majoritaire : `'2019/09/01'` et `'01/09/2019'`. Comme toutes les autres dates de la colonne ont pour jour `01` (jour fixe, probablement un enregistrement standardisé au 1er du mois), on en déduit que ces deux valeurs désignent la même date, le **1er septembre 2019**, simplement écrite avec un séparateur ou un ordre différent.

In [ ]:
# Correction des deux formats non-standard identifiés
pa['registration_date'] = pa['registration_date'].replace({
    '2019/09/01': '2019-09-01',
    '01/09/2019': '2019-09-01'
})

# Conversion en datetime, le format étant désormais uniforme
pa['registration_date'] = pd.to_datetime(pa['registration_date'], format='%Y-%m-%d')

# Vérification de cohérence : aucune inscription ne doit précéder la naissance
incoherent = pa[pa['registration_date'] < pa['date_of_birth']]
print(f"Nombre d'incohérences chronologiques détectées : {len(incoherent)}")
incoherent

### Synthèse du nettoyage — `patient_administration.csv`

| Opération | Résultat |
|---|---|
| Doublons supprimés | 1 ligne (241 → 240 lignes) |
| `record_status` normalisé | 3 catégories propres (`active`, `inactive`, `deceased`) |
| `home_region` normalisé | 5 régions propres |
| Âges vérifiés | Plage 23–91 ans, aucune aberration |
| Format de date corrigé | 2 valeurs sur `registration_date` ramenées au format `YYYY-MM-DD` |
| Cohérence chronologique | Aucune incohérence naissance/inscription détectée |

Le fichier nettoyé est sauvegardé pour être réutilisé dans les étapes suivantes du projet.

In [ ]:
# Sauvegarde du fichier nettoyé
pa.to_csv('data/health/patient_administration_clean.csv', index=False)

## Nettoyage — `clinical_procedures.csv`

Ce fichier recense les procédures médicales réalisées (type de procédure, horaires, autorisation, matériel utilisé, résultat). Il est lié à `clinical_encounters` via `encounter_number` et à `patient_administration` via `patient_number`.

**Problèmes attendus, à vérifier :**
1. Doublons de lignes
2. Incohérences de casse dans les variables catégorielles
3. Incohérence de formatage dans le texte libre (espaces superflus)
4. Formats de date hétérogènes (`scheduled_time`, `performed_time`)
5. Valeurs manquantes — à distinguer : absence structurelle vs vraie donnée manquante

In [ ]:
cp = pd.read_csv('data/health/clinical_procedures.csv')
print(f"Dimensions initiales : {cp.shape}")
cp.head()

### 1. Doublons

In [ ]:
doublons = cp[cp.duplicated(keep=False)]
print(f"Lignes impliquées dans un doublon : {len(doublons)}")
doublons

In [ ]:
cp = cp.drop_duplicates().reset_index(drop=True)
print(f"Dimensions après suppression : {cp.shape}")

### 2. Normalisation de la casse — variables catégorielles

`authorization_class` et `outcome_code` contiennent des variantes de casse pour les mêmes catégories (ex. `'standard'` / `'STANDARD'`), qui doivent être unifiées avant toute analyse par groupe.

In [ ]:
for col in ['procedure_code', 'authorization_class', 'procedure_status', 'outcome_code']:
    print(f"{col} -> {cp[col].unique()}")

In [ ]:
# Normalisation en minuscules
cp['authorization_class'] = cp['authorization_class'].str.lower()
cp['outcome_code'] = cp['outcome_code'].str.lower()

print(f"authorization_class -> {cp['authorization_class'].unique()}")
print(f"outcome_code -> {cp['outcome_code'].unique()}")

### 3. Nettoyage du texte libre — `procedure_description`

Cette colonne contient, pour un même `procedure_code`, des variantes avec espaces superflus en début/fin et entre les mots (ex. `'  Mobility  and  balance  panel '` au lieu de `'Mobility and balance panel'`). On uniformise avec un nettoyage d'espaces.

pour la suite de l'analyse : `procedure_description` est entièrement déterminé par `procedure_code` (relation 1 à 1) — c'est une redondance d'information au sens du cours.

In [ ]:
import re

# Suppression des espaces multiples et des espaces en début/fin de chaîne
cp['procedure_description'] = cp['procedure_description'].str.strip().apply(lambda s: re.sub(r'\s+', ' ', s))

# Vérification : chaque procedure_code ne doit plus avoir qu'une seule description
print(cp.groupby('procedure_code')['procedure_description'].unique())

### 4. Formats de date hétérogènes — `scheduled_time` / `performed_time`

On convertit ces deux colonnes en `datetime` et on isole les valeurs que pandas échoue à interpréter automatiquement, signe d'un format non standard.

In [ ]:
cp['scheduled_time_dt'] = pd.to_datetime(cp['scheduled_time'], errors='coerce')
cp['performed_time_dt'] = pd.to_datetime(cp['performed_time'], errors='coerce')

anomalies = cp[cp['scheduled_time_dt'].isna() | cp['performed_time_dt'].isna()]
anomalies[['procedure_record_id', 'scheduled_time', 'performed_time']]

Trois formats différents cohabitent dans ces colonnes : `YYYY-MM-DDTHH:MM:SS` (format majoritaire, ISO), `YYYY/MM/DD HH:MM:SS` (séparateur slash), et `DD-Mon-YYYY HH:MM:SS` (mois en texte abrégé). On les reconvertit un par un en forçant chaque format reconnu, plutôt que de laisser pandas deviner (ce qui a échoué pour ces 3 lignes précisément).

In [ ]:
def parse_mixed_datetime(s):
    for fmt in ('%Y-%m-%dT%H:%M:%S', '%Y/%m/%d %H:%M:%S', '%d-%b-%Y %H:%M:%S'):
        try:
            return pd.to_datetime(s, format=fmt)
        except ValueError:
            continue
    return pd.NaT  # aucun format reconnu

cp['scheduled_time'] = cp['scheduled_time'].apply(parse_mixed_datetime)
cp['performed_time'] = cp['performed_time'].apply(parse_mixed_datetime)

# On supprime les colonnes temporaires de diagnostic
cp = cp.drop(columns=['scheduled_time_dt', 'performed_time_dt'])

# Vérification : plus aucune valeur manquante après reconversion
print(f"Valeurs non converties - scheduled_time : {cp['scheduled_time'].isna().sum()}")
print(f"Valeurs non converties - performed_time : {cp['performed_time'].isna().sum()}")

# Vérification de cohérence logique : la procédure ne peut pas être réalisée avant d'être programmée
incoherent = cp[cp['performed_time'] < cp['scheduled_time']]
print(f"Incohérences performed < scheduled : {len(incoherent)}")

### 5. Valeurs manquantes — `material_reference` et `next_review_date`

Avant de décider quoi faire des valeurs manquantes, on regarde si elles sont réparties aléatoirement ou si elles suivent un motif, ce qui change complètement l'interprétation (cf. cours 1 — "l'absence d'information peut être une information en soi").

In [ ]:
# Répartition des valeurs manquantes de material_reference par type de procédure
print(cp.groupby('procedure_code')['material_reference'].apply(lambda s: f"{s.isna().sum()}/{len(s)} manquants"))

`material_reference` est manquant pour **100% des procédures**, sauf pour `MRP-42` (0% manquant, 50/50 renseigné). Ce n'est donc pas une donnée manquante aléatoire : seule la procédure `MRP-42` (administration d'un produit, probablement une injection) utilise un matériel traçable par référence. Pour les autres codes, l'absence est **structurelle et informative** (« pas de matériel utilisé » plutôt qu'une donnée perdue) — on ne l'impute pas, on la laisse telle quelle.

`next_review_date` est manquant entre 15% et 27% selon le type de procédure, de façon plus diffuse — ça ressemble davantage à « pas de suivi prévu pour ce patient » qu'à une erreur de saisie. Décision à documenter dans le rapport : on conserve ces valeurs manquantes telles quelles (elles seront interprétées comme « aucun suivi programmé » plutôt qu'imputées).

### Synthèse du nettoyage — `clinical_procedures.csv`

| Opération | Résultat |
|---|---|
| Doublons supprimés | 1 ligne (311 → 310 lignes) |
| `authorization_class`, `outcome_code` normalisés | Casse unifiée |
| `procedure_description` nettoyé | Espaces superflus supprimés (redondant avec `procedure_code`, à noter pour l'analyse) |
| `scheduled_time` / `performed_time` reconvertis | 3 formats de date hétérogènes unifiés en datetime, cohérence chronologique vérifiée |
| `material_reference` manquant | Conservé tel quel — absence structurelle (seule `MRP-42` en a) |
| `next_review_date` manquant | Conservé tel quel — interprété comme absence de suivi prévu |

In [ ]:
cp.to_csv('data/health/clinical_procedures_clean.csv', index=False)

## Nettoyage et analyse — `clinical_observations.csv`

Ce fichier contient les mesures individuelles prises à chaque rencontre clinique (une ligne = une mesure). On traite ce fichier en deux temps :

1. **Nettoyage**  : doublons, casse, formats de valeurs numériques
2. **Détection de patterns et d'anomalies**  : analyse univariée par IQR/Z-score modifié, puis recherche d'associations entre codes d'interprétation anormaux au sein d'une même rencontre (matrice de co-occurrence)

In [ ]:
obs = pd.read_csv('data/health/clinical_observations.csv')
print(f"Dimensions initiales : {obs.shape}")
obs.head()

In [ ]:
doublons = obs[obs.duplicated(keep=False)]
print(f"Lignes dupliquées : {len(doublons)}")
obs = obs.drop_duplicates().reset_index(drop=True)
print(f"Dimensions après suppression : {obs.shape}")

### 2. Harmonisation des catégories

`interpretation_code` et `quality_code` mélangent plusieurs casses pour une même catégorie (ex. `review` / `REVIEW`). On harmonise en minuscules.

In [ ]:
for col in ['interpretation_code', 'quality_code']:
    print(f"{col} avant -> {sorted(obs[col].unique())}")

obs['interpretation_code'] = obs['interpretation_code'].str.lower()
obs['quality_code'] = obs['quality_code'].str.lower()

for col in ['interpretation_code', 'quality_code']:
    print(f"{col} après -> {sorted(obs[col].unique())}")

### 3. Normalisation de `numeric_value`

Certaines valeurs ne se convertissent pas directement en nombre, pour deux raisons distinctes qu'on traite séparément :
- virgule décimale au lieu d'un point (`'84,9'`)
- unité collée au texte (`'82.2 index'`, `'8.07 L/min'`)

On garde la colonne brute intacte (traçabilité) et on construit une colonne numérique propre.

In [ ]:
# Étape a : repérer ce qui ne passe pas en numérique tel quel
valeur_numerique_brute = pd.to_numeric(obs['numeric_value'], errors='coerce')
a_corriger = obs['numeric_value'].notna() & valeur_numerique_brute.isna()
print(f"Valeurs à corriger : {a_corriger.sum()}")

# Étape b : on sépare la partie numérique de l'unité éventuelle, puis on gère la virgule
# Motif : un nombre (avec point OU virgule comme séparateur décimal), suivi ou non d'un texte d'unité
extraction = obs['numeric_value'].str.extract(r'^([-+]?\d+(?:[.,]\d+)?)')
nombre_point = extraction[0].str.replace(',', '.', regex=False)

obs['numeric_value_clean'] = pd.to_numeric(nombre_point, errors='coerce')

# Vérification : combien reste non converti après ce traitement ?
reste = obs['numeric_value'].notna() & obs['numeric_value_clean'].isna()
print(f"Valeurs encore non convertibles : {reste.sum()}")

In [ ]:
# Contrôle de cohérence : les valeurs corrigées restent-elles dans un ordre de grandeur plausible
# par rapport aux autres mesures du même observation_code ?
controle = obs.loc[a_corriged := a_corriger, ['observation_code', 'numeric_value', 'numeric_value_clean']]
display(controle.groupby('observation_code')['numeric_value_clean'].describe())

### 4. Détection d'anomalies univariée

Pour chaque `observation_code`on calcule les bornes IQR et le Z-score modifié (basé sur la médiane et l'ecart absolu median, moins sensible aux valeurs extrêmes que la moyenne/écart-type — cf. cours 3, diapo 14).

In [ ]:
import numpy as np

def detecter_outliers(groupe):
    valeurs = groupe.dropna()
    if len(valeurs) < 4:
        return pd.Series(dtype=float)

    # IQR
    q1, q3 = valeurs.quantile(.25), valeurs.quantile(.75)
    iqr = q3 - q1
    masque_iqr = (valeurs < q1 - 1.5 * iqr) | (valeurs > q3 + 1.5 * iqr)

    # Z-score modifié (médiane + écart absolu médian)
    mediane = valeurs.median()
    eam = (valeurs - mediane).abs().median()
    z_modifie = (valeurs - mediane) / eam if eam != 0 else pd.Series(0, index=valeurs.index)
    masque_z = z_modifie.abs() > 3.5  # seuil usuel pour le Z-score modifié

    return pd.DataFrame({
        'valeur': valeurs,
        'outlier_iqr': masque_iqr,
        'z_modifie': z_modifie.round(2),
        'outlier_z_modifie': masque_z
    })

resultats_outliers = {}
for code in obs['observation_code'].unique():
    sous_ensemble = obs[obs['observation_code'] == code]['numeric_value_clean']
    r = detecter_outliers(sous_ensemble)
    if len(r) > 0:
        resultats_outliers[code] = r

# Synthèse : nombre d'outliers détectés par méthode, par code
synthese_outliers = pd.DataFrame({
    code: {
        'n': len(r),
        'outliers_iqr': r['outlier_iqr'].sum(),
        'outliers_z_modifie': r['outlier_z_modifie'].sum()
    }
    for code, r in resultats_outliers.items()
}).T.astype(int)

synthese_outliers.sort_values('outliers_z_modifie', ascending=False)

### 5. Recherche de patterns — co-occurrence des interprétations anormales (cours 3)

On s'intéresse maintenant à un autre type de pattern : certaines mesures sont marquées `review` (à revoir) ou `discordant` (incohérente avec une autre mesure) plutôt qu'`within_expected_range`. On cherche si ces signalements se **produisent ensemble**, au sein d'une même rencontre, pour certains couples de type de mesure — ce qui serait un pattern au sens du cours 3 (itemsets, association).

**Construction des transactions :** chaque rencontre (`encounter_number`) est une transaction. Chaque `observation_code` est un item. Un item vaut 1 dans une transaction si cette mesure a été signalée `review` ou `discordant` lors de cette rencontre.

In [ ]:
# Construction de la matrice transactions x items
obs_flag = obs[obs['interpretation_code'].isin(['review', 'discordant'])]

matrice_items = (
    pd.crosstab(obs_flag['encounter_number'], obs_flag['observation_code'])
    .clip(upper=1)  # une mesure signalée plusieurs fois dans la même rencontre compte comme 1
)

print(f"Nombre de transactions (rencontres avec au moins un signalement) : {len(matrice_items)}")
print(f"Items (codes de mesure jamais signalés 'review'/'discordant') : {list(matrice_items.columns)}")
matrice_items.head()

In [ ]:
# Matrice de co-occurrence brute (cours 3, diapo 4), puis en pourcentage par ligne
co_occurrence = matrice_items.T.dot(matrice_items)
print("Matrice de co-occurrence (effectifs) :")
display(co_occurrence)

co_occurrence_pct = co_occurrence.div(np.diag(co_occurrence), axis=0) * 100
print("\nMatrice de co-occurrence (% par ligne) :")
display(co_occurrence_pct.round(1))

### 6. Évaluation des associations — support, confiance, lift (cours 3)

Pour chaque paire de codes, on calcule les mesures d'intérêt vues en cours : support (fréquence conjointe), confiance (P(Y|X)), et lift (force de l'association par rapport au hasard).

In [ ]:
from itertools import permutations

n_transactions = len(matrice_items)
support_individuel = matrice_items.mean()  # support de chaque item seul

regles = []
for x, y in permutations(matrice_items.columns, 2):
    support_xy = (matrice_items[x] & matrice_items[y]).mean()
    if support_xy == 0:
        continue
    confiance = support_xy / support_individuel[x]
    lift = confiance / support_individuel[y]
    regles.append({
        'X': x, 'Y': y,
        'support(X->Y)': round(support_xy, 3),
        'confiance(X->Y)': round(confiance, 3),
        'lift(X->Y)': round(lift, 3)
    })

df_regles = pd.DataFrame(regles).sort_values('lift(X->Y)', ascending=False)
df_regles

### Synthèse — `clinical_observations.csv`

| Étape | Résultat |
|---|---|
| Doublons supprimés | 2 lignes |
| Casse harmonisée | `interpretation_code`, `quality_code` |
| `numeric_value` normalisé | 340 valeurs corrigées (virgules décimales + unités collées au texte), colonne brute conservée |
| Outliers univariés | Détectés par IQR et Z-score modifié, par `observation_code` (voir tableau) |
| Pattern détecté | Les signalements `review`/`discordant` ne concernent que 5 codes (`APICAL`, `LATSWAY`, `OCULAR`, `THORPEAK`, `THORVIB`), sur 249/1017 rencontres |
| Association la plus forte | `APICAL` ↔ `THORPEAK` ↔ `THORVIB` forment un trio dont les anomalies co-occurrent (lift > 1.1 pour chaque paire) ; `OCULAR` et `LATSWAY` sont plus indépendants de ce trio (lift < 1 avec `THORPEAK`), à nuancer vu leur faible support |



In [ ]:
!git status